# CellPopAge

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.CellPopAge)

class CellPopAge(LinearReferenceClock):
    pass



In [3]:
model = pya.models.CellPopAge()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "cellpopage"
model.metadata["data_type"] = "DNA methylation"  # Paper: The model is based on DNA methylation measurements.
model.metadata["species"] = "Homo sapiens"  # Paper: The study samples are Homo sapiens.
model.metadata["year"] = 2024
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Lujan, Celia, et al. \"An expedited screening platform for the discovery of anti-ageing compounds in vitro and in vivo.\" Genome Medicine 16 (2024): 85."
model.metadata["doi"] = "https://doi.org/10.1186/s13073-024-01349-w"
model.metadata["notes"] = "CellPopAge is an elastic-net DNA-methylation clock using 42 selected CpGs to predict passage-based age of serially cultured adult primary human fibroblast populations and screen compounds that decelerate this measure."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["cultured fibroblasts"]  # Paper: The listed tissue is the model-development sample material.
model.metadata["predicts"] = ["cell-population passage age"]  # Paper: The reported predictor output is cell-population passage age.
model.metadata["training_target"] = ["cell passage number"]  # Paper: The fitting outcome is cell passage number.
model.metadata["unit"] = ["passages"]  # Paper: The returned construct is expressed as cell passage number.
model.metadata["model_type"] = "elastic net regression"  # Paper: The clock was fitted using Elastic net.
model.metadata["platform"] = ["Illumina EPIC"]  # Paper: Training/selection used Illumina EPIC.
model.metadata["population"] = "human cell cultures"  # Paper: adult primary human fibroblast cultures (39 training samples, passages 10–20)
model.metadata["journal"] = "Genome Medicine"
model.metadata["last_author"] = "Ivana Bjedov"
model.metadata["n_features"] = 42
model.metadata["citations"] = 6
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
os.system(f"curl -sL -o CellPopAge_ref.rda https://raw.githubusercontent.com/HigginsChenLab/methylCIPHER/19b12296b0d7eb7055a97d068064df635f44ce3e/data/CellPopAge_ref.rda")

0

In [6]:
%%writefile download.r

library(jsonlite)
load("CellPopAge_ref.rda")
write_json(CellPopAge_ref$all_CpGs, "coefficients.json", digits = 10)
write_json(CellPopAge_ref$intercept, "intercept.json", digits = 10)

Writing download.r


In [7]:
os.system("Rscript download.r")

0

## Load features

In [8]:
coef_df = pd.DataFrame(json.load(open('coefficients.json')))
model.features = coef_df['CpG'].tolist()
iv = json.load(open('intercept.json'))
intercept_value = iv[0] if isinstance(iv, list) else iv

## Load weights into base model

In [9]:
weights = torch.tensor(coef_df['coefficient'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([intercept_value]).float()

In [10]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [11]:
model.reference_values = None

## Load preprocess and postprocess objects

In [12]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [13]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [14]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Lujan, Gabriel M., et al. "A DNA methylation clock to measure '
             'the epigenetic age of cell populations." Genome Medicine 16 '
             '(2024).',
 'clock_name': 'cellpopage',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1186/s13073-024-01349-w',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2024}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg03662648', 'cg07230471', 'cg09689790', 'cg00283934', 'cg27617964', 'cg21734265', 'cg21184340', 'cg17342766', 'cg00312508', 'cg08907118', 'cg12071423', 'cg16622870', 'cg08126590', 'cg04121501', 'cg18571156', 'cg22900075', 'cg26475539', 'cg27500194', 'cg09992376', 'cg26450896', 'cg06863239', 'cg25651783'

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [16]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [17]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: intercept.json
Deleted file: download.r
Deleted file: coefficients.json
Deleted file: CellPopAge_ref.rda
